# 🏏 SupeRR Selector Season 2: Phase 4 — Reliability, Consistency & Unsupervised Clustering
## Separating Repeatable Clutch Performers from High-Variance Outliers

### What You Will Learn in This Phase:
1. **Consistency vs. Mean Impact**: Why two players with identical averages can have completely different match-by-match reliability.
2. **Unsupervised ML (K-Means Clustering + PCA)**: Discovering distinct tactical pressure archetypes directly from data embeddings.
3. **Confidence Intervals & Model Limitations**: Quantifying Standard Error ($SE$) across matches to document exactly where the model is most and least certain.

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

--- 
### Step 1: Loading the Full Qualified Rankings

In [ ]:
df_bat = pd.read_csv('../data/processed/ml_full_batters_under_pressure.csv')
df_bowl = pd.read_csv('../data/processed/ml_full_bowlers_under_pressure.csv')

print(f"Qualified Batters (>= 150 HP balls, >= 10 HP matches): {len(df_bat)}")
print(f"Qualified Bowlers (>= 30 HP overs, >= 15 HP matches): {len(df_bowl)}")

--- 
### Step 2: Visualizing Consistency vs. Clutch Value
Let's plot **Bayesian Match Consistency %** (Y-axis) against **Context-Adjusted Value Above Par** (X-axis), with point sizes showing total High-Pressure matches played.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Batters Plot
sns.scatterplot(data=df_bat, x='bayes_hp_rae', y='bayes_consistency_pct', size='hp_matches', hue='ml_archetype', sizes=(40, 300), alpha=0.8, ax=ax1)
for _, row in df_bat.head(7).iterrows():
    ax1.annotate(row['striker_name'], (row['bayes_hp_rae'], row['bayes_consistency_pct']), fontsize=8.5, fontweight='bold', xytext=(4, 4), textcoords='offset points')
ax1.set_title('Batters: Consistency vs. Runs Above Expected', fontweight='bold')
ax1.set_xlabel('Bayesian Pressure RAE per 100 Balls')
ax1.set_ylabel('Bayesian Match Consistency (%)')

# Bowlers Plot
sns.scatterplot(data=df_bowl, x='bayes_hp_rsae', y='bayes_consistency_pct', size='hp_matches', hue='ml_archetype', sizes=(40, 300), alpha=0.8, ax=ax2)
for _, row in df_bowl.head(7).iterrows():
    ax2.annotate(row['bowler_name'], (row['bayes_hp_rsae'], row['bayes_consistency_pct']), fontsize=8.5, fontweight='bold', xytext=(4, 4), textcoords='offset points')
ax2.set_title('Bowlers: Consistency vs. Runs Saved Above Expected', fontweight='bold')
ax2.set_xlabel('Bayesian Pressure RSAE per Over')
ax2.set_ylabel('Bayesian Match Consistency (%)')

plt.tight_layout()
plt.show()

--- 
### Step 3: PCA Visualization of K-Means Clutch Archetypes
We trained an Unsupervised K-Means Clustering model (`k=3`) to automatically group players into tactical archetypes.
Using **Principal Component Analysis (PCA)**, we can project the multi-dimensional feature space down to 2D to visually confirm clean cluster separation.

In [ ]:
features = ['bayes_hp_rae', 'bayes_clutch_delta', 'hp_boundary_pct', 'bayes_survival']
X_scaled = StandardScaler().fit_transform(df_bat[features].fillna(0))
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)
df_bat['pca_1'] = coords[:, 0]
df_bat['pca_2'] = coords[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_bat, x='pca_1', y='pca_2', hue='ml_archetype', s=100, alpha=0.85, palette='Set1')
for _, row in df_bat.head(10).iterrows():
    plt.annotate(row['striker_name'], (row['pca_1'], row['pca_2']), fontsize=8.5, fontweight='bold', xytext=(4, 4), textcoords='offset points')
plt.title('Unsupervised ML Archetypes (K-Means projected via 2D PCA) - Batters', fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

--- 
### Step 4: Uncertainty Quantification & Model Limitations
Where is our model most certain vs. least certain? Let's inspect Standard Errors (`uncertainty_se`) across Top 10 Batters and Bowlers.
Players with fewer high-pressure matches (e.g., Dewald Brevis & Naman Dhir at 13 matches) carry wider confidence intervals than high-volume fixtures like Suryakumar Yadav (26 matches), Rovman Powell (29 matches), and Jasprit Bumrah (40 matches).

In [ ]:
print("=== Top 10 Batters Confidence & Sample Bounds ===")
display(df_bat.head(10)[['rank', 'striker_name', 'hp_matches', 'hp_balls', 'bayes_hp_rae', 'uncertainty_se', 'batting_clutch_score']])

print("\n=== Top 10 Bowlers Confidence & Sample Bounds ===")
display(df_bowl.head(10)[['rank', 'bowler_name', 'hp_matches', 'hp_overs', 'bayes_hp_rsae', 'uncertainty_se', 'bowling_clutch_score']])